In [ ]:
def var_clause_plot(data: dict, file_name: str = {GRAPH_DIR}):
    plt.figure(figsize=(12, 5))

    # Variables
    plt.subplot(1, 2, 1)
    plt.plot(data['bit_size'], data['vars_simplified'], label='Simplifié', marker='o')
    plt.plot(data['bit_size'], data['vars_unsimplified'], label='Non simplifié', marker='x')
    plt.xlabel("Taille (bits)")
    plt.ylabel("Nombre de variables")
    plt.title("Variables vs taille des entrées [bits]")
    plt.legend()

    # Clauses
    plt.subplot(1, 2, 2)
    plt.plot(data['bit_size'], data['clauses_simplified'], label='Simplifié', marker='o')
    plt.plot(data['bit_size'], data['clauses_unsimplified'], label='Non simplifié', marker='x')
    plt.xlabel("Taille (bits)")
    plt.ylabel("Nombre de clauses")
    plt.title("Clauses vs taille des entrées [bits]")
    plt.legend()

    plt.tight_layout()
    plt.savefig(f"{file_name}/varClauses.png")
    
    

def plot_boxplots(data_dict: dict, label: str, xlabel: str = "Taille des entrées (bits)", 
                    ylabel: str = "Temps de résolution (s)", title: str = "Temps de résolution SAT par taille d'entrée"):
    plt.figure(figsize=(12, 6))
    keys = sorted(data_dict.keys())
    values = [data_dict[k] for k in keys]
    plt.boxplot(values, positions=keys, widths=0.6, patch_artist=True, label=label)
    
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{GRAPH_DIR}/boxplots/boxplot_{label}.png")



def plot_linearplot(filepath: str, data_dict: dict, label: str, xlabel: str = "Taille des entrées (bits)", 
                    ylabel: str = "Temps de résolution (s)", title: str = "Temps de résolution SAT par taille d'entrée"):
    plt.figure(figsize=(12, 6))
    keys = sorted(data_dict.keys())
    values = [np.mean(data_dict[k]) for k in keys]  # Moyenne pour chaque clé

    plt.plot(keys, values, marker='o', label=label)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(filepath)
    
    
def plot_linearplot2(data_dict_list: list[dict], label_list: list[str], xlabel: str = "Taille des entrées (bits)", 
                    ylabel: str = "Temps de résolution (s)", title: str = "Temps de résolution SAT par taille d'entrée"):
    plt.figure(figsize=(12, 6))
    for i, dicti in enumerate(data_dict_list):
        keys = sorted(dicti.keys())
        values = [np.mean(dicti[k]) for k in keys]  # Moyenne pour chaque clé

        plt.plot(keys, values, marker='o', label=label_list[i])
        plt.xlabel(xlabel)
        plt.ylabel(ylabel)
        plt.title(title)
        plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(f"{GRAPH_DIR}/linearplots/linearplot_all.png")
    
    
    import matplotlib.pyplot as plt
import numpy as np
import os

In [12]:
class TseytinTransformation:
    
    def __init__(self):
        self.num = 0  
        self.sat = ""   
        self.dimacs = ""
        
        
        
    def AND(self, A: str, B: str, etage, niveau, num = 0) -> str:
        gate = f"g{etage}-{niveau}-AND{num}"
        res = f"(¬{A} + ¬{B} + {gate}) * ({A} + ¬{gate}) * ({B} + ¬{gate})"
        return res, gate
    
    
    
    def OR(self, A: str, B: str, etage, niveau) -> str:
        gate = f"g{etage}-{niveau}-OR"
        res = f"({A} + {B} + ¬{gate}) * (¬{A} + {gate}) * (¬{B} + {gate})"
        return res, gate
    
    
    
    def XOR(self, A: str, B: str, etage, niveau, num = 0) -> str:
        gate = f"g{etage}-{niveau}-XOR{num}"
        res = f"(¬{A} + ¬{B} + ¬{gate}) * ({A} + {B} + ¬{gate}) * ({A} + ¬{B} + {gate}) * (¬{A} + {B} + {gate})"
        return res, gate
    
    
    # HA
    def half_ADD(self, A: str, B: str, etage, niveau) -> str:
        m1, g1 = self.XOR(A, B, etage, niveau, self.num)
        m2, g2 = self.AND(A, B, etage, niveau, self.num)
        
        res = m1 + " * " + m2
        
        return (res, g1, g2)
    
    
    # FA
    def full_ADD(self, A: str, B: str, c_in: str, etage, niveau) -> str:
        m1, g1 = self.XOR(A, B, etage, niveau, self.num)
        m2, g2 = self.XOR(g1, c_in, etage, niveau, self.num) #correspond à la sortie z de l'additionneur
        m3, g3 = self.AND(g1, c_in, etage, niveau, self.num)
        m4, g4 = self.AND(A, B, etage, niveau, self.num)
        m5, g5 = self.OR(g3, g4, etage, niveau)
        
        res = m1 + " * " + m2 + " * " + m3 + " * " + m4 + " * " + m5 

        return (res, g2, g5)
    
    
    
    def MULT(self, size_x: int, size_y: int) -> str:
        """
        Ajoute à la variable 'sat' la formule SAT correspondant à x * y = z

        Args:
            size_x, size_y (int):
                taille des entrées x et y en bits 
                
        """
        
        X, Y = [f"x{i}" for i in range(size_x)], [f"y{i}" for i in range(size_y)]
        Z = []
        res, gate = self.AND(X[0], Y[0], 0, 0, 0) # "étage 0": première multiplication dont on récupère directement le résultat en sortie
        self.sat += res
        
        for i in range(size_y-1): # longueur des facteurs moins 1, correspond à l'indice des B
            for j in range(size_x): # correspond à l'indice des A
                if i == 0: # 1ère étage
                    if j < size_x - 1:
                        m1, g1 = self.AND(X[j+1], Y[i], i, j, self.num)
                        m2, g2 = self.AND(X[j], Y[i+1], i, j, self.num)
                        
                        res, n, c_out = self.half_ADD(g1, g2, i, j) if j == 0 else self.full_ADD(g1, g2, c_out, i, j)
                        Z.append(n)
                        
                        self.sat += " * " + m1 + " * " + m2 + " * " + res
                        
                    else:
                        m, g = self.AND(X[j], Y[i+1], i, j,self.num)
                        
                        res, n, c_out = self.half_ADD(c_out, g, i, j)
                        Z.append(n)
                        Z.append(c_out)
                        
                        self.sat += " * " + m + " * " + res
                        
                else:
                    
                    m, g = self.AND(X[j], Y[i+1], i, j, self.num)
                    res, n, c_out = self.half_ADD(Z[j], g, i, j) if j == 0 else self.full_ADD(Z[j], g, c_out, i, j)
                    Z[j] = n 
                    if j == size_x - 1:
                        Z.append(c_out)
                    
                    self.sat += " * " + m + " * " + res 
                    
            Z.pop(0) #permet de supprimer le resultat qui sera la sortie et donc plus nécessaire pour la suite des calculs

In [13]:
ts = TseytinTransformation()
ts.MULT(4, 4)
print(ts.sat)

(¬x0 + ¬y0 + g0-0-AND0) * (x0 + ¬g0-0-AND0) * (y0 + ¬g0-0-AND0) * (¬x1 + ¬y0 + g0-0-AND0) * (x1 + ¬g0-0-AND0) * (y0 + ¬g0-0-AND0) * (¬x0 + ¬y1 + g0-0-AND0) * (x0 + ¬g0-0-AND0) * (y1 + ¬g0-0-AND0) * (¬g0-0-AND0 + ¬g0-0-AND0 + ¬g0-0-XOR0) * (g0-0-AND0 + g0-0-AND0 + ¬g0-0-XOR0) * (g0-0-AND0 + ¬g0-0-AND0 + g0-0-XOR0) * (¬g0-0-AND0 + g0-0-AND0 + g0-0-XOR0) * (¬g0-0-AND0 + ¬g0-0-AND0 + g0-0-AND0) * (g0-0-AND0 + ¬g0-0-AND0) * (g0-0-AND0 + ¬g0-0-AND0) * (¬x2 + ¬y0 + g0-1-AND0) * (x2 + ¬g0-1-AND0) * (y0 + ¬g0-1-AND0) * (¬x1 + ¬y1 + g0-1-AND0) * (x1 + ¬g0-1-AND0) * (y1 + ¬g0-1-AND0) * (¬g0-1-AND0 + ¬g0-1-AND0 + ¬g0-1-XOR0) * (g0-1-AND0 + g0-1-AND0 + ¬g0-1-XOR0) * (g0-1-AND0 + ¬g0-1-AND0 + g0-1-XOR0) * (¬g0-1-AND0 + g0-1-AND0 + g0-1-XOR0) * (¬g0-1-XOR0 + ¬g0-0-AND0 + ¬g0-1-XOR0) * (g0-1-XOR0 + g0-0-AND0 + ¬g0-1-XOR0) * (g0-1-XOR0 + ¬g0-0-AND0 + g0-1-XOR0) * (¬g0-1-XOR0 + g0-0-AND0 + g0-1-XOR0) * (¬g0-1-XOR0 + ¬g0-0-AND0 + g0-1-AND0) * (g0-1-XOR0 + ¬g0-1-AND0) * (g0-0-AND0 + ¬g0-1-AND0) * (¬g0-1-A

In [ ]:
def AND(X: str, Y: str):
    if len(X) > len(Y):
        to_add = len(X) - len(Y)
        Y = "0" * to_add + Y
    elif len(X) < len(Y):
        to_add = len(Y) - len(X)
        X = "0" * to_add + X
    
    Z = ""
    for x, y in zip(X[::-1], Y[::-1]):
        if x == "1" and y == "1":
            Z = "1" + Z
        else:
            Z = "0" + Z 
    
    return Z

In [ ]:
def OR(X, Y):
    if len(X) > len(Y):
        to_add = len(X) - len(Y)
        Y = "0" * to_add + Y
    elif len(X) < len(Y):
        to_add = len(Y) - len(X)
        X = "0" * to_add + X
        
    Z = ""
    for x, y in zip(X[::-1], Y[::-1]):
        if x == "1" or y == "1":
            Z = "1" + Z
        else:
            Z = "0" + Z 
    
    return Z

In [ ]:
def XOR(X, Y):
    
    if len(X) > len(Y):
        to_add = len(X) - len(Y)
        Y = "0" * to_add + Y
    elif len(X) < len(Y):
        to_add = len(Y) - len(X)
        X = "0" * to_add + X
    
    
    Z = ""
    for x, y in zip(X[::-1], Y[::-1]):
        if x == y:
            Z = "0" + Z
        else:
            Z = "1" + Z 
    
    return Z

In [ ]:
def half_ADD(x: str, y: str):
    z = XOR(x, y)
    c_out = AND(x, y)
    return (z, c_out)

In [ ]:
def full_ADD(x: str, y: str, c_in: str):
    tmp = XOR(x, y)
    z = XOR(tmp, c_in)
    c_out = OR(AND(tmp, c_in), AND(x, y))
    return (z, c_out)

In [ ]:
def ADD(X: str, Y: str):
    if len(X) > len(Y):
        to_add = len(X) - len(Y)
        Y = "0" * to_add + Y
    elif len(X) < len(Y):
        to_add = len(Y) - len(X)
        X = "0" * to_add + X
        
    c_in = None
    res = ""
    for x, y in zip(X[::-1], Y[::-1]):
        if c_in == None:
            z, c_out = half_ADD(x, y)
        else:
            z, c_out = full_ADD(x, y, c_in)
            
        res = z + res
        c_in = c_out
        
    if c_out == "1":
        res = c_out + res
        
    return res

In [ ]:
def MULT(X: str, Y: str):
    inter = []
    res = ""
    
    for i, y in enumerate(Y[::-1]):
        dec = i * "0"
        if y == "1":
            tmp = X + dec
            inter.append(X + dec)
        else:
            inter.append("0" * (len(X) + len(dec)))
    
    for i in range(len(inter)):
        res = ADD(res, inter[i])
        
    return res

In [ ]:
class CNFToDIMACS:
    def __init__(self, formula: str):
        self.formula = formula
        self.var_map = {} 
        self.clauses = []  
        self.var_counter = 1  

    def parse_formula(self):
        cleaned_formula = self.formula.replace(" ", "") # supprimer les espaces
        raw_clauses = cleaned_formula.split("*")

        for raw_clause in raw_clauses:
            clause = []
            content = raw_clause.strip("()") #  enlever les parenthèses
            literals = content.split("+")
            for lit in literals:
                is_neg = False
                if lit.startswith("¬"):
                    is_neg = True
                    lit = lit.strip("¬")
                if lit not in self.var_map: # gérer la négation
                    self.var_map[lit] = self.var_counter
                    self.var_counter += 1
                var_num = self.var_map[lit]
                clause.append(-var_num if is_neg else var_num)
            self.clauses.append(clause)

    def to_dimacs(self, output_file: str):
        self.parse_formula() 
        num_vars = len(self.var_map)
        num_clauses = len(self.clauses)

        with open(output_file, 'w') as f:
            f.write(f"p cnf {num_vars} {num_clauses}\n")
            for clause in self.clauses:
                line = " ".join(map(str, clause)) + " 0\n"
                f.write(line)

In [ ]:
def find_output_gate(k: int) -> list:
    outputs = []
    gate = 0
    for i in range(k):
        
        if i == 0:
            gate += 3                       # 2 portes A0 et B0 + AND
            outputs.append(gate)
        
        elif i == 1 and i != k:
            gate += 2 + 2 + 1               # 2 portes: une A1 et B1 + 2*AND + 1ère porte HADD
            outputs.append(gate)
            gate += 1                       # 2ème porte HADD
            gate += (1 + 2 + 5) * (k - 2)   # [porte Ai + 2*AND + ADD] k-2 fois
            gate += 1 + 2                   # porte B + HADD
            
        elif i > 1 and i != k-1:
            gate += 1 + 1 + 1               # porte Bi + AND + 1ère porte HADD
            outputs.append(gate)
            gate += 1                       # 2ème porte HADD
            gate += (1 + 5) * (k - 1)       # [AND + ADD] * k-1 fois
            
        else:
            gate += 1 + 1 + 1               # porte Bi + AND + 1ère porte HADD
            outputs.append(gate)
            gate += 1                       # 2ème porte HADD
            for _ in range(k - 1):
                gate += 1 + 2               # AND + 2 premières porte du ADD
                outputs.append(gate)
                gate += 3                   # 3 dernières portes 
            outputs.append(gate)        
    
    return outputs

In [ ]:
def add_z_to_dimacs(z: int, dimacs: str): ### Faudrait que je passe directe en binaire
    z_bin = dec_to_bin(z)
    k = len(z_bin)  
    k = optimal_k(z)
    z_var = find_output_gate(k)
    while len(z_bin) != len (z_var):
        z_bin = "0" + z_bin
        
    with open(dimacs, "a") as f:
        for zi_bin, zi_var in zip(z_bin[::-1], z_var):
            f.write(f"{str(zi_var)} 0\n") if zi_bin == "1" else f.write(f"-{str(zi_var)} 0\n")